<a href="https://colab.research.google.com/github/ahs95/sentiment-analysis-cwcbd23/blob/main/sentiment_sarcasm_bdcwc23_focal_loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install iterative-stratification
!pip install --upgrade datasets
!pip install bitsandbytes
!pip install gradio

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import copy
import gradio as gr
from collections import Counter

In [ ]:
import warnings
warnings.filterwarnings('ignore')
nltk.download('omw-1.4', quiet=True)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (17,7)
plt.rcParams['font.size'] = 18

In [ ]:
import torch
import bitsandbytes as bnb
from torch.utils.data import DataLoader, Dataset
from transformers import get_cosine_schedule_with_warmup, AutoModel, AutoTokenizer
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay
from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from tqdm import tqdm
from sklearn.utils import resample

In [ ]:
def optimize_thresholds(logits, labels, threshold_step=0.05):
    """
    Find optimal per-class thresholds that maximize Macro F1.
    logits: (N, C) tensor of raw model outputs
    labels: (N,) array/tensor of true class indices
    """
    probs = F.softmax(logits, dim=1).cpu().numpy()
    labels_np = np.array(labels.cpu() if torch.is_tensor(labels) else labels)
    num_classes = probs.shape[1]
    thresholds_grid = np.arange(0.05, 0.95, threshold_step)
    opt_thresholds = np.zeros(num_classes)

    for c in range(num_classes):
        best_f1 = 0.0
        best_t = 0.5
        y_true = (labels_np == c).astype(int)

        for t in thresholds_grid:
            y_pred = (probs[:, c] >= t).astype(int)
            if y_pred.sum() == 0 or y_true.sum() == 0:
                continue
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        opt_thresholds[c] = best_t
    return opt_thresholds

def predict_with_thresholds(logits, thresholds):
    """
    Vectorized prediction with per-class thresholds.
    Falls back to standard argmax if no class exceeds its threshold.
    """
    probs = F.softmax(logits, dim=1)
    thresholds_t = torch.tensor(thresholds, device=logits.device).unsqueeze(0)
    valid_mask = (probs >= thresholds_t).float()
    masked_probs = probs * valid_mask

    has_valid = valid_mask.sum(dim=1) > 0
    preds = torch.argmax(masked_probs, dim=1)
    preds[~has_valid] = torch.argmax(probs, dim=1)[~has_valid]
    return preds

In [ ]:
data = pd.read_excel('/content/drive/MyDrive/Data_Cric23_BD.xlsx')
data

In [ ]:
data.drop('source', axis=1, inplace=True)

In [ ]:
data.info()

In [ ]:
print(data.isnull().sum())

In [ ]:
data.shape

In [ ]:
data['sentiment'].value_counts()

In [ ]:
data['sarcasm'].value_counts()

In [ ]:
out_dir = "/content/drive/MyDrive/plots"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Distribution of sentiment
plt.figure(figsize=(10, 5))
sns.histplot(x='sentiment', data=data, discrete=True)
plt.title('Distribution of Sentiment')
plt.savefig(os.path.join(out_dir, "sen-dis.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of sarcasm
plt.figure(figsize=(10, 5))
sns.histplot(x='sarcasm', data=data, discrete=True)
plt.title('Distribution of Sarcasm')
plt.savefig(os.path.join(out_dir, "sar-dis.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
Counter(" ".join(data["text"]).split()).most_common(200)

In [ ]:
def clean_text(text):
    # Convert to string just in case
    text = str(text)
    # Remove URLs/HTML if noisy
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['text'] = data['text'].apply(clean_text)

In [ ]:
data.head(10)

In [ ]:
data.text.str.len().max()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert_small")

# Create a Dataset object
dataset = Dataset.from_pandas(data)

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        max_length=512,
        truncation=True
    )

# Map tokenization across the dataset (uses multiprocessing)
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=64,
    num_proc=4
)

In [ ]:
class DualHeadModel(nn.Module):
    def __init__(self, num_sentiment_classes=4, num_sarcasm_classes=2):
        super(DualHeadModel, self).__init__()
        self.bert = AutoModel.from_pretrained("csebuetnlp/banglabert_small")
        hidden_size = self.bert.config.hidden_size

        self.hidden = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.sentiment_classifier = nn.Linear(256, num_sentiment_classes)
        self.sarcasm_classifier = nn.Linear(256, num_sarcasm_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        features = self.hidden(cls_output)
        sentiment_logits = self.sentiment_classifier(features)
        sarcasm_logits = self.sarcasm_classifier(features)
        return sentiment_logits, sarcasm_logits

In [ ]:
sentiment_order = ['Positive', 'Neutral', 'Negative', 'Mixed']
sarcasm_order = ['Sarcastic', 'Non-Sarcastic']

# 1. Clean & standardize strings (handles extra spaces & case mismatches)
data['sentiment'] = data['sentiment'].astype(str).str.strip().str.title()
data['sarcasm'] = data['sarcasm'].astype(str).str.strip().str.title()

# 2. Map to integers
data['sentiment_enc'] = data['sentiment'].map({lbl: i for i, lbl in enumerate(sentiment_order)})
data['sarcasm_enc'] = data['sarcasm'].map({lbl: i for i, lbl in enumerate(sarcasm_order)})

# 3. SAFETY CHECK: Will immediately tell you if mapping failed
if data['sentiment_enc'].isna().any() or data['sarcasm_enc'].isna().any():
    print("⚠️ WARNING: Label mapping failed! Check exact values in your Excel file.")
    print("Sentiment unique:", data['sentiment'].unique())
    print("Sarcasm unique:", data['sarcasm'].unique())

In [ ]:
labels_sent = pd.get_dummies(data['sentiment_enc']).astype(int).reindex(columns=range(4), fill_value=0).values
labels_sarc = pd.get_dummies(data['sarcasm_enc']).astype(int).reindex(columns=range(2), fill_value=0).values
labels = np.hstack([labels_sent, labels_sarc])

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def focal_loss(logits, labels, alpha=None, gamma=2.0):
    ce_loss = F.cross_entropy(logits, labels, reduction='none')
    p_t = torch.exp(-ce_loss)

    if alpha is not None:
        alpha_t = alpha[labels].to(logits.device)
    else:
        alpha_t = torch.ones_like(labels).float().to(logits.device)

    focal = alpha_t * (1 - p_t) ** gamma * ce_loss
    return focal.mean()

def custom_loss_fn(sentiment_logits, sarcasm_logits,
                   sentiment_labels, sarcasm_labels,
                   sent_alpha=None, sarc_alpha=None, gamma=2.0):
    sent_loss = focal_loss(sentiment_logits, sentiment_labels, alpha=sent_alpha, gamma=gamma)
    sarc_loss = focal_loss(sarcasm_logits, sarcasm_labels, alpha=sarc_alpha, gamma=gamma)
    return sent_loss + sarc_loss

In [ ]:
class MetricEarlyStopping:
    def __init__(self, patience=2, mode='max', min_delta=0.001):
        """
        Early stopping callback for model training.

        Args:
            patience (int): Number of epochs with no improvement before stopping.
            mode (str): 'max' for metrics like F1/Accuracy, 'min' for Loss.
            min_delta (float): Minimum change to qualify as an improvement.

        Usage:
            early_stopper = MetricEarlyStopping(patience=3, mode='max')
            for epoch in range(num_epochs):
                # ... training code ...
                early_stopper(validation_score, model)
                if early_stopper.early_stop:
                    print("Early stopping triggered!")
                    break
            early_stopper.restore_best_weights(model)
        """
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, current_score, model):
        """
        Check if metric improved and update best model accordingly.

        Args:
            current_score (float): Current validation metric value.
            model: PyTorch model to potentially save.
        """
        if self.best_score is None:
            self.best_score = current_score
            self.save_checkpoint(model)
            print(f"🎯 Initial metric: {self.best_score:.4f}")
        else:
            if self.mode == 'max':
                score_improved = current_score > self.best_score + self.min_delta
            else:
                score_improved = current_score < self.best_score - self.min_delta

            if score_improved:
                self.best_score = current_score
                self.save_checkpoint(model)
                self.counter = 0
                print(f"✅ Metric improved to: {self.best_score:.4f}")
            else:
                self.counter += 1
                print(f"⚠️ Metric hasn't improved. Best: {self.best_score:.4f}. Counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
                    print(f"🛑 Early stopping triggered after {self.patience} epochs without improvement.")

    def save_checkpoint(self, model):
        """Save a deep copy of model weights."""
        self.best_model_state = copy.deepcopy(model.state_dict())

    def restore_best_weights(self, model):
        """Restore the best model weights found during training."""
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)
            print(f"✅ Restored best model weights (Metric: {self.best_score:.4f}).")
        else:
            print("⚠️ No checkpoint to restore. Run at least one epoch first.")

In [ ]:
# ==================== K-FOLD TRAINING LOOP ====================
print("Preparing data for training...")
kf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_for_split = np.array(tokenized_dataset['input_ids'])
folds = kf.split(X_for_split, labels)

all_fold_texts, all_fold_sentiment_true, all_fold_sentiment_pred = [], [], []
all_fold_sarcasm_true, all_fold_sarcasm_pred, fold_history = [], [], []
all_fold_sentiment_logits, all_fold_sarcasm_logits = [], []

accumulation_steps, num_epochs = 2, 5

for fold, (train_index, val_index) in enumerate(folds):
    print(f"\n{'='*50}\nTraining Fold {fold + 1}/{5}\n{'='*50}")
    GAMMA_MAX, GAMMA_MIN, ALPHA_MIN, ALPHA_MAX = 2.5, 0.8, 0.15, 0.45

    early_stopping = MetricEarlyStopping(patience=2, mode='max', min_delta=0.001)
    model = DualHeadModel().to(device)
    enc_keys = [k for k in tokenized_dataset.column_names if k.startswith(('input', 'attention', 'token_type'))]
    train_enc = {k: np.array(tokenized_dataset[k])[train_index] for k in enc_keys}
    val_enc = {k: np.array(tokenized_dataset[k])[val_index] for k in enc_keys}
    train_labels, val_labels = labels[train_index], labels[val_index]

    print(f"Fold {fold + 1} Sent Dist: {np.sum(train_labels[:, :4], axis=0)}")
    print(f"Fold {fold + 1} Sarc Dist: {np.sum(train_labels[:, 4:], axis=0)}")

    train_loader = DataLoader(CustomDataset(train_enc, train_labels), shuffle=True, batch_size=16)
    val_loader = DataLoader(CustomDataset(val_enc, val_labels), batch_size=16)

    # 🔹 COMPUTE PER-CLASS ALPHA (Inverse Frequency)
    sent_counts = torch.tensor(np.sum(train_labels[:, :4], axis=0), dtype=torch.float)
    sarc_counts = torch.tensor(np.sum(train_labels[:, 4:], axis=0), dtype=torch.float)
    sent_inv_freq = sent_counts.sum() / (4 * (sent_counts + 1e-8))
    sarc_inv_freq = sarc_counts.sum() / (2 * (sarc_counts + 1e-8))
    sent_alpha = torch.clamp(ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * (sent_inv_freq / sent_inv_freq.max()), ALPHA_MIN, ALPHA_MAX).to(device)
    sarc_alpha = torch.clamp(ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * (sarc_inv_freq / sarc_inv_freq.max()), ALPHA_MIN, ALPHA_MAX).to(device)

    optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-5)
    steps_per_epoch = len(train_loader) // accumulation_steps
    lr_scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_epochs * steps_per_epoch)

    for epoch in range(num_epochs):
        current_gamma = GAMMA_MAX - (GAMMA_MAX - GAMMA_MIN) * (epoch / max(num_epochs - 1, 1))
        model.train()
        total_train_loss = 0

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            sent_logits, sarc_logits = model(batch["input_ids"], batch["attention_mask"])
            sent_labels_b = torch.argmax(batch["labels"][:, :4], dim=1)
            sarc_labels_b = torch.argmax(batch["labels"][:, 4:], dim=1)

            loss = custom_loss_fn(sent_logits, sarc_logits, sent_labels_b, sarc_labels_b, sent_alpha, sarc_alpha, current_gamma)
            loss = loss / accumulation_steps
            loss.backward()

            if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
            total_train_loss += loss.item() * accumulation_steps

        avg_train_loss = total_train_loss / len(train_loader)

        # VALIDATION
        model.eval()
        total_val_loss, sent_preds, sent_trues, sarc_preds, sarc_trues = 0, [], [], [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                sent_logits, sarc_logits = model(batch["input_ids"], batch["attention_mask"])
                sent_labels_b = torch.argmax(batch["labels"][:, :4], dim=1)
                sarc_labels_b = torch.argmax(batch["labels"][:, 4:], dim=1)
                total_val_loss += custom_loss_fn(sent_logits, sarc_logits, sent_labels_b, sarc_labels_b, sent_alpha, sarc_alpha, current_gamma).item()
                sent_preds.extend(torch.argmax(sent_logits, dim=1).cpu().numpy())
                sent_trues.extend(sent_labels_b.cpu().numpy())
                sarc_preds.extend(torch.argmax(sarc_logits, dim=1).cpu().numpy())
                sarc_trues.extend(sarc_labels_b.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_loader)
        epoch_senti_f1 = f1_score(sent_trues, sent_preds, average='macro')
        epoch_sarcasm_f1 = f1_score(sarc_trues, sarc_preds, average='macro')
        current_metric = (0.6 * epoch_sarcasm_f1) + (0.4 * epoch_senti_f1)

        print(f"Epoch {epoch + 1}/{num_epochs} - Train: {avg_train_loss:.4f} - Val: {avg_val_loss:.4f} - Comp: {current_metric:.4f} (S:{epoch_sarcasm_f1:.4f}, N:{epoch_senti_f1:.4f})")
        early_stopping(current_metric, model)
        if early_stopping.early_stop:
            print("🛑 Early stopping triggered.")
            break

        fold_history.append({'fold': fold + 1, 'epoch': epoch + 1, 'train_loss': avg_train_loss, 'val_loss': avg_val_loss,
                             'senti_f1': epoch_senti_f1, 'sarcasm_f1': epoch_sarcasm_f1, 'composite_score': current_metric})

    # FINAL VALIDATION WITH BEST WEIGHTS
    early_stopping.restore_best_weights(model)
    model.eval()
    fold_sent_logits, fold_sarc_logits = [], []
    best_sent_preds, best_sarc_preds = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            s_log, sarc_log = model(batch["input_ids"], batch["attention_mask"])
            best_sent_preds.extend(torch.argmax(s_log, dim=1).cpu().numpy())
            best_sarc_preds.extend(torch.argmax(sarc_log, dim=1).cpu().numpy())
            fold_sent_logits.append(s_log.cpu())
            fold_sarc_logits.append(sarc_log.cpu())

    all_fold_texts.extend(data.iloc[val_index]['text'].tolist())
    all_fold_sentiment_true.extend(val_labels[:, :4].argmax(axis=1))
    all_fold_sentiment_pred.extend(best_sent_preds)
    all_fold_sarcasm_true.extend(val_labels[:, 4:].argmax(axis=1))
    all_fold_sarcasm_pred.extend(best_sarc_preds)
    all_fold_sentiment_logits.extend(fold_sent_logits)
    all_fold_sarcasm_logits.extend(fold_sarc_logits)
    print(f"✓ Fold {fold + 1} Complete | Accumulated: {len(all_fold_sentiment_true)}")

In [ ]:
# ==================== THRESHOLD TUNING & EVALUATION ====================
print("\n🔍 Tuning optimal decision thresholds...")
all_fold_sentiment_logits = torch.cat(all_fold_sentiment_logits, dim=0)
all_fold_sarcasm_logits = torch.cat(all_fold_sarcasm_logits, dim=0)
all_fold_sentiment_true = torch.tensor(all_fold_sentiment_true)
all_fold_sarcasm_true = torch.tensor(all_fold_sarcasm_true)

sent_thresholds = optimize_thresholds(all_fold_sentiment_logits, all_fold_sentiment_true)
sarc_thresholds = optimize_thresholds(all_fold_sarcasm_logits, all_fold_sarcasm_true)
print(f"📊 Sentiment Thresholds: {np.round(sent_thresholds, 2)}")
print(f"📊 Sarcasm Thresholds:   {np.round(sarc_thresholds, 2)}")

all_fold_sentiment_pred = predict_with_thresholds(all_fold_sentiment_logits, sent_thresholds).cpu().numpy()
all_fold_sarcasm_pred = predict_with_thresholds(all_fold_sarcasm_logits, sarc_thresholds).cpu().numpy()

print(f"\n📊 Sentiment Weighted F1: {f1_score(all_fold_sentiment_true, all_fold_sentiment_pred, average='weighted'):.4f}")
print(f"📊 Sarcasm Weighted F1:   {f1_score(all_fold_sarcasm_true, all_fold_sarcasm_pred, average='weighted'):.4f}")

target_names_sent = ['Positive', 'Neutral', 'Negative', 'Mixed']
target_names_sarc = ['Sarcastic', 'Non-Sarcastic']

In [ ]:
print("\n📈 Sentiment Report:\n", classification_report(all_fold_sentiment_true, all_fold_sentiment_pred, target_names=target_names_sent))
print("\n📈 Sarcasm Report:\n", classification_report(all_fold_sarcasm_true, all_fold_sarcasm_pred, target_names=target_names_sarc))

In [ ]:
def compute_bootstrap_ci(y_true, y_pred, average='weighted', n_boots=2000, confidence=0.95, seed=42):
    f1_scores = []  # Use a list to avoid indexing bugs

    for i in tqdm(range(n_boots), desc="Bootstrapping", leave=False):
        y_true_boot, y_pred_boot = resample(y_true, y_pred, n_samples=len(y_true), replace=True, random_state=i)
        f1_scores.append(f1_score(y_true_boot, y_pred_boot, average=average))

    f1_scores = np.array(f1_scores)
    lower = np.percentile(f1_scores, 100 * (1 - confidence) / 2)
    upper = np.percentile(f1_scores, 100 * (1 - (1 - confidence) / 2))
    return np.mean(f1_scores), lower, upper

# ==================== COMPUTE CIs ====================
sent_mean, sent_lower, sent_upper = compute_bootstrap_ci(
    all_fold_sentiment_true, all_fold_sentiment_pred, average='weighted'
)
sarc_mean, sarc_lower, sarc_upper = compute_bootstrap_ci(
    all_fold_sarcasm_true, all_fold_sarcasm_pred, average='weighted'
)

print(f"\n📊 BOOTSTRAPPED 95% CONFIDENCE INTERVALS")
print(f"Sentiment F1: {sent_mean:.4f} [{sent_lower:.4f} – {sent_upper:.4f}]")
print(f"Sarcasm F1:   {sarc_mean:.4f} [{sarc_lower:.4f} – {sarc_upper:.4f}]")

In [ ]:
sentiment_cm = confusion_matrix(all_fold_sentiment_true, all_fold_sentiment_pred)
sarcasm_cm = confusion_matrix(all_fold_sarcasm_true, all_fold_sarcasm_pred)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(sentiment_cm, display_labels=target_names_sent).plot(ax=ax1, cmap=plt.cm.Blues, values_format='d')
ax1.set_title('Sentiment Confusion Matrix')
fig1.savefig(os.path.join(out_dir, "confusion_sentiment.png"), dpi=300, bbox_inches='tight')
plt.show()

fig2, ax2 = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(sarcasm_cm, display_labels=target_names_sarc).plot(ax=ax2, cmap=plt.cm.Blues, values_format='d')
ax2.set_title('Sarcasm Confusion Matrix')
fig2.savefig(os.path.join(out_dir, "confusion_sarcasm.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== PREPARE HISTORY DATAFRAME ====================
hist_df = pd.DataFrame(fold_history)
last_fold = hist_df['fold'].max()
last_fold_data = hist_df[hist_df['fold'] == last_fold].reset_index(drop=True)

# ==================== LOSS CURVE ====================
fig_loss, ax_loss = plt.subplots(figsize=(10, 5))

ax_loss.plot(last_fold_data['epoch'], last_fold_data['train_loss'],
             label='Train Loss', linewidth=2, marker='o', color='tab:blue')
ax_loss.plot(last_fold_data['epoch'], last_fold_data['val_loss'],
             label='Val Loss', linewidth=2, marker='s', color='tab:red')

ax_loss.set_xlabel('Epoch', fontsize=12)
ax_loss.set_ylabel('Loss', fontsize=12)
ax_loss.set_title(f'Loss Curve (Fold {last_fold})', fontsize=14, fontweight='bold')
ax_loss.legend(loc='best', fontsize=11)
ax_loss.grid(True, linestyle='--', alpha=0.6)

save_path_loss = '/content/drive/MyDrive/learning_curve_loss.png'
fig_loss.tight_layout()
plt.savefig(save_path_loss, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== ERROR ANALYSIS ====================
sentiment_map = {0: 'Positive', 1: 'Neutral', 2: 'Negative', 3: 'Mixed'}
sarcasm_map = {0: 'Sarcastic', 1: 'Non-Sarcastic'}

# Force 1D lists of native Python ints to prevent dtype mismatches
true_sent = np.array(all_fold_sentiment_true).flatten().astype(int).tolist()
pred_sent = np.array(all_fold_sentiment_pred).flatten().astype(int).tolist()
true_sarc = np.array(all_fold_sarcasm_true).flatten().astype(int).tolist()
pred_sarc = np.array(all_fold_sarcasm_pred).flatten().astype(int).tolist()

error_df = pd.DataFrame({
    'Text': all_fold_texts,
    'True_Sentiment': [sentiment_map.get(c, 'Unknown') for c in true_sent],
    'Pred_Sentiment': [sentiment_map.get(c, 'Unknown') for c in pred_sent],
    'True_Sarcasm': [sarcasm_map.get(c, 'Unknown') for c in true_sarc],
    'Pred_Sarcasm': [sarcasm_map.get(c, 'Unknown') for c in pred_sarc]
})

error_df['Sentiment_Correct'] = error_df['True_Sentiment'] == error_df['Pred_Sentiment']
error_df['Sarcasm_Correct'] = error_df['True_Sarcasm'] == error_df['Pred_Sarcasm']
errors_only = error_df[~(error_df['Sentiment_Correct'] & error_df['Sarcasm_Correct'])].copy()

def classify_error(row):
    if not row['Sentiment_Correct'] and not row['Sarcasm_Correct']:
        return 'Both Wrong'
    elif not row['Sentiment_Correct']:
        return 'Sentiment Wrong'
    else:
        return 'Sarcasm Wrong'

errors_only['Error_Type'] = errors_only.apply(classify_error, axis=1)

print(f"\n✅ Total Val Samples: {len(error_df)} | Errors: {len(errors_only)} | Error Rate: {(len(errors_only)/len(error_df))*100:.2f}%")
print(errors_only['Error_Type'].value_counts())

# Save
output_path = '/content/drive/MyDrive/model_error_analysis.xlsx'
errors_only[['Text', 'True_Sentiment', 'Pred_Sentiment', 'True_Sarcasm', 'Pred_Sarcasm', 'Error_Type']].to_excel(output_path, index=False)

print(f"\n{'='*50}")
print("✅ COMPREHENSIVE ERROR ANALYSIS (ALL 5 FOLDS)")
print(f"{'='*50}")
print(f"📊 Total Validation Samples: {len(error_df)}")
print(f"⚠️  Total Errors Found: {len(errors_only)}")
print(f"📈 Error Rate: {(len(errors_only)/len(error_df))*100:.2f}%")
print(f"💾 File saved to: {output_path}")
print(f"\n{'='*50}")
print("ERROR TYPE DISTRIBUTION:")
print(f"{'='*50}")
print(errors_only['Error_Type'].value_counts())

In [ ]:
# ==================== SAVE MODEL & THRESHOLDS ====================
save_dir = "/content/drive/MyDrive/sentiment_sarcasm_model"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_dir, "model.pth"))
tokenizer.save_pretrained(save_dir)
np.save(os.path.join(save_dir, "sent_thresholds.npy"), sent_thresholds)
np.save(os.path.join(save_dir, "sarc_thresholds.npy"), sarc_thresholds)
print(f"💾 Model & Thresholds saved to {save_dir}")

In [ ]:
# ==================== GRADIO INTERFACE ====================
tokenizer = AutoTokenizer.from_pretrained(save_dir)
model = DualHeadModel()
model.load_state_dict(torch.load(os.path.join(save_dir, "model.pth"), map_location=device))
model.to(device).eval()

sent_thresholds = np.load(os.path.join(save_dir, "sent_thresholds.npy"))
sarc_thresholds = np.load(os.path.join(save_dir, "sarc_thresholds.npy"))
sentiment_labels = {0:"Positive", 1:"Neutral", 2:"Negative", 3:"Mixed"}
sarcasm_labels = {0:"Sarcastic", 1:"Non-Sarcastic"}

def classify_text(text):
    try:
        if not text or not str(text).strip():
            return "No input provided", "No input provided"
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            sent_logits, sarc_logits = model(inputs["input_ids"], inputs["attention_mask"])
        sent_pred = predict_with_thresholds(sent_logits, sent_thresholds).item()
        sarc_pred = predict_with_thresholds(sarc_logits, sarc_thresholds).item()
        return sentiment_labels.get(sent_pred, "Unknown"), sarcasm_labels.get(sarc_pred, "Unknown")
    except Exception as e:
        return f"Error: {str(e)}", "Error"

gr.Interface(
    fn=classify_text, inputs="text",
    outputs=[gr.Textbox(label="Sentiment Prediction"), gr.Textbox(label="Sarcasm Prediction")],
    title="Bangla Cricket Comment Analyzer",
    description="Predicts sentiment and sarcasm in social media comments related to the Bangladesh cricket team."
).launch()